# Santander Customer Transaction Prediction — Feature Engineering Challenge

### **Goal:** Improve baseline Recall (≈0.83-0.88) through systematic feature engineering — 
### frequency encoding, row-wise stats, interactions, and outlier handling.

### **Constraints:** ONLY LOGISITIC REGRESSION. No outside data. 

### **Data:** Fetched automatically via kagglehub at runtime ,not stored in repo because the data is 268+ MB it exceeds the github limit
 



---

## KAGGLE API KEY - SETUP

## WHY KAGGLE API ?

### ---THE GIVEN DATA SET IS LARGE IT EXCEEDS THE LIMIT OF THE GITHUB STORAGE, INSTEAD OF MANUALLLY DOWNLOADING THE DATASET AND PUSHING INTO GITHUB 

### --- WE ARE SETTING UP A LOCAL KAGGLE API TOKEN FROM LOCAL **.ENV** FILE AND ACCESS USING THE **python-dotenv**

### --- I HAVE GIVEN THE **.ENV** FILE FORMAT IN THE FILE NAMED **.ENV.SAMPLE** 

## IMPLEMENTATION OF LOADING THE APIKEY FROM THE ENV

In [46]:
import os
from dotenv import load_dotenv

#loads the variable from the .env
load_dotenv()

#storing the apikey into the variable
token = os.getenv("KAGGLE_API_TOKEN")

#if the apikey is missing , we are raising a error
if not token:
    raise ValueError(
        " KAGGLE_API_TOKEN not found. "
        "Copy .env.sample to .env and add your Kaggle API token before proceeding."
    )

print(" Kaggle API token loaded successfully.")

 Kaggle API token loaded successfully.


## LOADING/IMPORTING THE DATA 

### --With the token loaded, **kagglehub** fetches the competition dataset directly into its own cache outside this repo. This avoids the 260MB+ file size limit on GitHub

### --And loading the dataset using the **Pandas** Library **pd.read_csv("path")** and storing it into the variable **df**

In [47]:
import kagglehub
import pandas as pd

# Downloads dataset to kagglehub's cache (not stored in repo)
path = kagglehub.competition_download("santander-customer-transaction-prediction")
print("Dataset downloaded to:", path)

# Confirm what's inside
import os
print("Files:", os.listdir(path))

# Load the training data
train_path = f"{path}/train.csv"
df = pd.read_csv(train_path)

print("Shape:", df.shape)
df.head()

Dataset downloaded to: /Users/pranusaravanan/.cache/kagglehub/competitions/santander-customer-transaction-prediction
Files: ['test.csv', 'train.csv', 'sample_submission.csv']
Shape: (200000, 202)


,ID_code,target,var_0,var_1,var_2,var_3,var_4,var_5,var_6,var_7,...,var_190,var_191,var_192,var_193,var_194,var_195,var_196,var_197,var_198,var_199
0,train_0,0,8.9255,-6.7863,11.9081,5.0930,11.4607,-9.2834,5.1187,18.6266,...,4.4354,3.9642,3.1364,1.6910,18.5227,-2.3978,7.8784,8.5635,12.7803,-1.0914
1,train_1,0,11.5006,-4.1473,13.8588,5.3890,12.3622,7.0433,5.6208,16.5338,...,7.6421,7.7214,2.5837,10.9516,15.4305,2.0339,8.1267,8.7889,18.3560,1.9518
2,train_2,0,8.6093,-2.7457,12.0805,7.8928,10.5825,-9.0837,6.9427,14.6155,...,2.9057,9.7905,1.6704,1.6858,21.6042,3.1417,-6.5213,8.2675,14.7222,0.3965
3,train_3,0,11.0604,-2.1518,8.9522,7.1957,12.5846,-1.8361,5.8428,14.9250,...,4.4666,4.7433,0.7178,1.4214,23.0347,-1.2706,-2.9275,10.2922,17.9697,-8.9996
4,train_4,0,9.8369,-1.4834,12.8746,6.6375,12.2772,2.4486,5.9405,19.2514,...,-1.4905,9.5214,-0.1508,9.1942,13.2876,-1.5121,3.9267,9.5031,17.9974,-8.8104


## TRAIN_TEST_SPLIT

### WE SPLIT RANDOMLY THE DATASET INTO X_Train,X_Test,Y_Test,Y_Train

### WHERE X_Train,X_Test ---> INPUT FEATURES

### WHERE Y_Train,X_Test ---> OUTPUT FEATURES

### --AFTER LOADING THE DATA , BEFORE ANY EDA WE DO THE TRAIN_TEST_SPLIT INORDER TO REDUCE DATALEAKAGE

### --IF WE DO FEATURE ENGNEERING BEFORE THE TRAIN_TEST_SPLIT , WE ACCIDENTLY LET THE INFORMATION FROM THE DATA THAT WE USE FOR THE TEST LATER

### IMPLEMENTATION

In [48]:
from sklearn.model_selection import train_test_split

# Separate features and target first
X = df.drop(columns=["target", "ID_code"])  # drop target + ID column
y = df["target"]

#SPLITTING THE DATA INTO 85% TRAIN , 15% TEST 
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.15, stratify=y, random_state=42
)

#SPLITTING THE 85% TRAIN TO 70% TRAIN , 15% VALIDATION
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.15, stratify=y_train_val, random_state=42
)

print("Train:", X_train.shape, y_train.shape)
print("Val:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

Train: (144500, 200) (144500,)
Val: (25500, 200) (25500,)
Test: (30000, 200) (30000,)


## EXPERIMENT LOGGING

### --INSTEAD OF LOGGING THE VALUES INTO THE CSV FILE MANUALLY , WE ARE DEFINING A FUNCTION **LOG_EXPREMENT** WHICH AUTOMATICALLY UPDATES INTO CSV ONCE IT CALLS

In [49]:
import csv
import os

log_path = "../logs/experiment_log.csv"

def log_experiment(exp_id, description, recall, precision=None, f1=None, notes=""):
    file_exists = os.path.exists(log_path)
    with open(log_path, "a", newline="") as f:
        writer = csv.writer(f)
        if not file_exists:
            writer.writerow(["experiment", "description", "recall", "precision", "f1", "notes"])
        writer.writerow([exp_id, description, recall, precision, f1, notes])

## BASELINE MODEL

### WITHOUT ANY FEATURE ENGNEERING PURE LOGISTIC REGRESSION GIVE'S US RECALL=0.77 (APPROX)

### TRAINING A LOGISTIC REGRESSION MODEL WITHOUT ANY FEATURE ENGNEERING

In [50]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_score, f1_score


scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

baseline_model = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
baseline_model.fit(X_train_scaled, y_train)

y_val_pred = baseline_model.predict(X_val_scaled)
precision = precision_score(y_val, y_val_pred)
f1 = f1_score(y_val, y_val_pred)

print("Baseline Recall:", recall_score(y_val, y_val_pred))
print("Precision:", precision, "F1:", f1)


Baseline Recall: 0.7712724434035909
Precision: 0.28374497415278577 F1: 0.41486458114633634


/Users/pranusaravanan/acm-week1-2-tasks/feature-engineering-challenge/venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
/Users/pranusaravanan/acm-week1-2-tasks/feature-engineering-challenge/venv/lib/python3.14/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/pranusaravanan/acm-week1-2-tasks/feature-engineering-challenge/venv/lib/python3.14/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/pranusaravanan/acm-week1-2-tasks/feature-engineering-challenge/venv/lib/python3.14/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/pranusaravanan/acm-week1-2-tasks/feature-engineering

## EXPLORATORY DATA ANALYSIS (EDA)

### --WE DO THIS EDA ONLY ON X_train, y_train — NEVER ON X_val OR X_test

### --IF WE LOOK AT VAL/TEST DATA EVEN JUST TO "EXPLORE", WE QUIETLY LEAK INFORMATION AND DEFEAT THE PURPOSE OF LOCKING IT AWAY

### STEP 0 - Check Dataset Quality

### Before exploring the data further, verify that the training data is clean. Check for missing values, duplicate rows, and confirm that every feature has the expected numeric data type. These quick sanity checks help ensure that later feature engineering and model training are built on reliable data.

### In this dataset, no missing values or duplicate rows are expected, but performing these checks is still a standard part of the EDA process.

In [51]:
# Missing values
print("Missing values per column:")
print(X_train.isnull().sum().sort_values(ascending=False).head())

print(f"\nTotal missing values: {X_train.isnull().sum().sum()}")

# Duplicate rows
print(f"\nDuplicate rows: {X_train.duplicated().sum()}")

# Data types
print("\nFeature data types:")
print(X_train.dtypes.value_counts())

Missing values per column:
var_0      0
var_137    0
var_127    0
var_128    0
var_129    0
dtype: int64

Total missing values: 0

Duplicate rows: 0

Feature data types:
float64    200
Name: count, dtype: int64



### --No missing values or duplicate rows were found, and all predictor columns are numeric (float64). Since the training data is already clean, no preprocessing is required before continuing with the exploratory analysis.

## STEP 1 — CHECK CLASS BALANCE

In [52]:
# checking how balanced the two target classes are
print(y_train.value_counts())
print(y_train.value_counts(normalize=True))

target
0    129979
1     14521
Name: count, dtype: int64
target
0    0.899509
1    0.100491
Name: proportion, dtype: float64


### --CLASS IMBALANCE CONFIRMED: ~90% CLASS 0, ~10% CLASS 1, THIS IS WHY RECALL AND class_weight='balanced' MATTER MORE THAN ACCURACY HERE

## STEP 2 — GLANCE AT A FEW INDIVIDUAL COLUMNS

In [53]:
# quick statistical summary of a few sample columns
X_train[["var_0", "var_1", "var_81", "var_139"]].describe()

,var_0,var_1,var_81,var_139
count,144500.000000,144500.000000,144500.000000,144500.000000
mean,10.685524,-1.635552,14.716036,7.765100
std,3.036229,4.051899,2.301991,7.692048
min,0.408400,-14.091000,7.997200,-21.274300
25%,8.460075,-4.751025,13.210200,2.387125
50%,10.528400,-1.621800,14.843500,8.072000
75%,12.767600,1.346700,16.337300,13.244300
max,20.315000,10.376800,23.132400,35.552700


### --INDIVIDUAL COLUMN CORRELATIONS ARE VERY WEAK (STRONGEST IS ~0.08), NO SINGLE COLUMN IS A STRONG PREDICTOR ON ITS OWN

## STEP 3 — RANK ALL COLUMNS BY CORRELATION WITH TARGET

### --WE TEMPORARILY COMBINE X_train AND y_train JUST TO CALCULATE CORRELATION, THIS IS STILL SAFE BECAUSE WE ONLY USE THE TRAIN SET

In [54]:
# combining X_train and y_train temporarily to compute correlation
train_corr = X_train.copy()
train_corr["target"] = y_train

# ranking every column by how strongly it relates to the target
correlations = train_corr.corr()["target"].drop("target").abs().sort_values(ascending=False)

correlations.head(20)

var_81     0.081649
var_139    0.075993
var_12     0.069859
var_6      0.065204
var_110    0.065015
var_146    0.064288
var_76     0.064026
var_174    0.061602
var_53     0.061149
var_26     0.060981
var_22     0.060191
var_21     0.059387
var_80     0.058054
var_190    0.057258
var_166    0.056907
var_99     0.056386
var_148    0.055454
var_2      0.055358
var_13     0.054843
var_165    0.054690
Name: target, dtype: float64

### --THIS IS EXACTLY WHY FEATURE ENGINEERING MATTERS FOR THIS DATASET, RAW COLUMNS ALONE DON'T CARRY MUCH SIGNAL

## STEP 4 — SAVE THE SHORTLIST

### --WE KEEP THE TOP 20 STRONGEST COLUMNS, WE REUSE THIS EXACT LIST  FOR INTERACTION FEATURES"

In [55]:
top_features = correlations.head(20).index.tolist()
print("Top 20 correlated features:", top_features)

Top 20 correlated features: ['var_81', 'var_139', 'var_12', 'var_6', 'var_110', 'var_146', 'var_76', 'var_174', 'var_53', 'var_26', 'var_22', 'var_21', 'var_80', 'var_190', 'var_166', 'var_99', 'var_148', 'var_2', 'var_13', 'var_165']


## FEATURE ENGINEERING — FREQUENCY ENCODING

### --FOR EACH COLUMN, WE ADD A NEW COLUMN THAT SAYS HOW COMMON THAT PARTICULAR VALUE IS, THIS IS USUALLY THE SINGLE BIGGEST IMPROVEMENT ON THIS DATASET

### --THE MOST IMPORTANT RULE HERE: WE CALCULATE THE FREQUENCY MAP USING ONLY X_train, THEN APPLY THE SAME MAP TO X_val AND X_test

### --IF WE CALCULATE FREQUENCIES USING ALL DATA MIXED TOGETHER, OUR SCORE WILL LOOK BETTER THAN IT ACTUALLY IS AND WON'T HOLD UP LATER WHILE VALIDATION

### IMPLEMENTATION

In [56]:
# building frequency encoding, one new column per original column
def add_frequency_encoding(df, freq_maps=None, is_train=False):
    df = df.copy()
    
    if is_train:
        freq_maps = {}
    
    for col in df.columns:
        if is_train:
            # learning the frequency map only from this (train) data
            freq_maps[col] = df[col].value_counts(normalize=True)
        
        # mapping each value to how common it is, unseen values get 0
        df[col + "_freq"] = df[col].map(freq_maps[col]).fillna(0)
    
    return df, freq_maps

# fitting the frequency map on X_train only
X_train_freq, freq_maps = add_frequency_encoding(X_train, is_train=True)

# applying the same train-derived map to val and test, not refitting
X_val_freq, _ = add_frequency_encoding(X_val, freq_maps=freq_maps, is_train=False)
X_test_freq, _ = add_frequency_encoding(X_test, freq_maps=freq_maps, is_train=False)

print("Train shape after freq encoding:", X_train_freq.shape)
print("Val shape after freq encoding:", X_val_freq.shape)

/var/folders/gr/ytk6s4zs22g0srn5bl73c85r0000gn/T/ipykernel_95895/343190651.py:14: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df[col + "_freq"] = df[col].map(freq_maps[col]).fillna(0)
/var/folders/gr/ytk6s4zs22g0srn5bl73c85r0000gn/T/ipyker

Train shape after freq encoding: (144500, 400)
Val shape after freq encoding: (25500, 400)


/var/folders/gr/ytk6s4zs22g0srn5bl73c85r0000gn/T/ipykernel_95895/343190651.py:14: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df[col + "_freq"] = df[col].map(freq_maps[col]).fillna(0)
/var/folders/gr/ytk6s4zs22g0srn5bl73c85r0000gn/T/ipyker

### RETRAIN THE MODEL AND CHECK SCORE

In [57]:
# scaling the new feature set
scaler_freq = StandardScaler()
X_train_freq_scaled = scaler_freq.fit_transform(X_train_freq)
X_val_freq_scaled = scaler_freq.transform(X_val_freq)

# retraining logistic regression on train + frequency features
freq_model = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
freq_model.fit(X_train_freq_scaled, y_train)

y_val_pred_freq = freq_model.predict(X_val_freq_scaled)

recall_freq = recall_score(y_val, y_val_pred_freq)
precision_freq = precision_score(y_val, y_val_pred_freq)
f1_freq = f1_score(y_val, y_val_pred_freq)

print("Recall after frequency encoding:", recall_freq)
print("Precision:", precision_freq, "F1:", f1_freq)

/Users/pranusaravanan/acm-week1-2-tasks/feature-engineering-challenge/venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
/Users/pranusaravanan/acm-week1-2-tasks/feature-engineering-challenge/venv/lib/python3.14/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/pranusaravanan/acm-week1-2-tasks/feature-engineering-challenge/venv/lib/python3.14/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/pranusaravanan/acm-week1-2-tasks/feature-engineering-challenge/venv/lib/python3.14/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/pranusaravanan/acm-week1-2-tasks/feature-engineering

Recall after frequency encoding: 0.9863387978142076
Precision: 0.1256089074460682 F1: 0.2228395061728395


/Users/pranusaravanan/acm-week1-2-tasks/feature-engineering-challenge/venv/lib/python3.14/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/pranusaravanan/acm-week1-2-tasks/feature-engineering-challenge/venv/lib/python3.14/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/pranusaravanan/acm-week1-2-tasks/feature-engineering-challenge/venv/lib/python3.14/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


### LOG THIS EXPERIMENT

In [58]:
log_experiment(
    exp_id="exp_02_freq_encoding",
    description="Added frequency encoding for all 200 columns",
    recall=recall_freq,
    precision=precision_freq,
    f1=f1_freq,
    notes="Baseline was 0.7712 recall, checking improvement from freq features"
)

### --RECALL JUMPED TO 0.9863 BUT PRECISION COLLAPSED TO 0.1256 AND F1 DROPPED FROM 0.4148 TO 0.2228

### --THIS IS NOT A REAL IMPROVEMENT, THE MODEL IS LIKELY JUST PREDICTING "1" FOR MOST ROWS, HIGH RECALL ALONE IS MEANINGLESS IF PRECISION FALLS THIS HARD

### --WE NEED TO CHECK PREDICTED VS ACTUAL CLASS DISTRIBUTION TO SEE IF THE MODEL IS GENUINELY LEARNING OR JUST PREDICTING "1" MOST OF THE TIME TO GAME RECALL

In [59]:
# checking what the model is actually predicting
import numpy as np
print("Predicted class distribution:", np.bincount(y_val_pred_freq))
print("Actual class distribution:", np.bincount(y_val))


Predicted class distribution: [ 5382 20118]
Actual class distribution: [22938  2562]


### --IT IS CONFIRMED THAT PREDICTED [5382, 20118] BUT ACTUAL IS [22938, 2562], MODEL IS PREDICTING CLASS 1 FOR ~79% OF ROWS WHEN ONLY ~10% ACTUALLY ARE

### --THIS MEANS THE HIGH RECALL IS FAKE, THE MODEL ISN'T LEARNING REAL SIGNAL, IT'S JUST OVER-PREDICTING CLASS 1, LIKELY BECAUSE class_weight='balanced' PUSHED THE DECISION BOUNDARY TOO FAR ALONGSIDE THE NEW FREQUENCY FEATURES

## RETRY FREQUENCY ENCODING WITHOUT class_weight='balanced'

In [62]:
# retraining frequency-encoded features, this time without class_weight='balanced'
freq_model_v2 = LogisticRegression(max_iter=1000, random_state=42)
freq_model_v2.fit(X_train_freq_scaled, y_train)

y_val_pred_freq_v2 = freq_model_v2.predict(X_val_freq_scaled)

recall_freq_v2 = recall_score(y_val, y_val_pred_freq_v2)
precision_freq_v2 = precision_score(y_val, y_val_pred_freq_v2)
f1_freq_v2 = f1_score(y_val, y_val_pred_freq_v2)

print("Recall (no class_weight):", recall_freq_v2)
print("Precision:", precision_freq_v2, "F1:", f1_freq_v2)
print("Predicted class distribution:", np.bincount(y_val_pred_freq_v2))
log_experiment(exp_id="exp_03_freq_no_classweight",description="Frequency encoding without class_weight='balanced'",recall=recall_freq_v2,precision=precision_freq_v2,f1=f1_freq_v2,
    notes="Fixed degenerate model from exp_02. F1 still slightly below baseline (0.3977 vs 0.4148). class_weight='balanced' was the real cause of the earlier fake recall, not the freq features themselves."
)

Recall (no class_weight): 0.8040593286494926
Precision: 0.2641703000769428 F1: 0.39768339768339767
Predicted class distribution: [17702  7798]


/Users/pranusaravanan/acm-week1-2-tasks/feature-engineering-challenge/venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
/Users/pranusaravanan/acm-week1-2-tasks/feature-engineering-challenge/venv/lib/python3.14/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/pranusaravanan/acm-week1-2-tasks/feature-engineering-challenge/venv/lib/python3.14/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/pranusaravanan/acm-week1-2-tasks/feature-engineering-challenge/venv/lib/python3.14/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/pranusaravanan/acm-week1-2-tasks/feature-engineering

### --REMOVING class_weight='balanced' FIXED IT , PRECISION RECOVERED TO 0.2641, PREDICTED CLASS 1 COUNT DROPPED FROM 20118 TO 7798 

### --HOWEVER F1 (0.3977) IS STILL SLIGHTLY BELOW BASELINE (0.4148), SO FREQUENCY ENCODING ALONE ISN'T YET A CLEAR IMPROVEMENT, THE EARLIER "FAKE" RECALL WAS PURELY FROM class_weight='balanced' DISTORTING THE DECISION BOUNDARY